# modification
1) unfreeze last 2 blocks other than FCL
2) fedprox instead of fed avg
additonally

1) unfreezing 6th layer
2) added class weights balancer in the loss
3) added cosine scheduler
4) the lr is set by the server side optimizer so it will change only after the rounds not within each epochs so modifiying it to local optimizer

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [1]:
import os
import copy
from collections import OrderedDict, Counter
import numpy as np
from collections import OrderedDict
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset
from collections import defaultdict
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

from sklearn.metrics import classification_report

In [2]:
class Net(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        # Load the base EfficientNet model with pre-trained weights
        self.base = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)

        # 1. Freeze the entire backbone initially
        for p in self.base.parameters():
            p.requires_grad = False

        # 2. PARTIAL UNFREEZE: Unfreeze the last two blocks (features.7 and features.8)
        # EfficientNet features are in blocks (0 to 8). We unfreeze the last few.
        # Unfreeze features.7 (Block 7)
        for p in self.base.features[6].parameters():
            p.requires_grad = True

        for p in self.base.features[7].parameters():
            p.requires_grad = True

        # Unfreeze features.8 (Block 8, which contains the final convolution layer)
        for p in self.base.features[8].parameters():
            p.requires_grad = True

        # 3. Replace and Unfreeze the final fully connected (FC) classifier
        num_ftrs = self.base.classifier[-1].in_features
        self.base.classifier[-1] = nn.Linear(num_ftrs, num_classes)

        for p in self.base.classifier.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.base(x)

In [3]:
img_tf = Compose([
    Resize((256, 256)),
    ToTensor(),
    Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

In [4]:
DATA_ROOT = "/kaggle/input/distributed-dataset/Splitted_data"
BATCH_SIZE = 32
N_FOLDS = 5
SEED = 42

def load_client_data(client_id):
    path = os.path.join(DATA_ROOT, f"Client_{client_id}")
    dataset = ImageFolder(path, transform=img_tf)

    n = len(dataset)
    t = int(0.8 * n)
    v = n - t

    train_ds, val_ds = random_split(dataset, [t, v], generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)
    return train_loader, val_loader

def get_client_dataset(client_id):
    path = os.path.join(DATA_ROOT, f"Client_{client_id}")
    dataset = ImageFolder(path, transform=img_tf)
    return dataset

In [5]:
def get_stratified_folds(dataset, n_splits=N_FOLDS, seed=SEED):
    # ImageFolder stores labels as dataset.targets on some torchvision versions, otherwise extract
    if hasattr(dataset, "targets"):
        labels = np.array(dataset.targets)
    else:
        labels = np.array([dataset[i][1] for i in range(len(dataset))])

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = list(skf.split(np.zeros(len(labels)), labels))
    # folds is list of (train_idx, val_idx)
    return folds

In [6]:
def average_state_dicts(state_dicts):
    avg = copy.deepcopy(state_dicts[0])
    n = len(state_dicts)
    for k in avg.keys():
        # accumulate
        for i in range(1, n):
            avg[k] = avg[k] + state_dicts[i][k]
        avg[k] = avg[k] / n
    return avg

In [7]:
def local_train(model, loader, epochs, lr, device):
    model.to(device)
    model.train()

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()

    return model.state_dict()  # return updated weights

In [8]:
from sklearn.metrics import classification_report

def local_eval(model, data_loader, device,class_weights):
    model.eval()
    model.to(device)
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)

    # Generate a detailed classification report
    report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)

    return avg_loss, report

In [9]:
def local_train_fedprox(model, loader, epochs, lr, mu, device, class_weights):
    model.to(device)
    model.train()

    # Deepcopy the global model weights w^t for the proximal term calculation
    w_global = OrderedDict(copy.deepcopy(model.state_dict()))

    # 1. Instantiate Optimizer
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    # 2. Instantiate Local Scheduler
    # T_max is set to the number of local epochs for decay across the local training session
    local_scheduler = CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-7)

    # 3. Use the provided class_weights in CrossEntropyLoss
    crit = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()

            # --- A. Standard Loss (Cross-Entropy) ---
            loss = crit(model(x), y)

            # --- B. Proximal Term (FedProx) ---
            proximal_term = 0.0

            # Iterate through all local model parameters (w)
            for name, param in model.named_parameters():
                if name in w_global:
                    # Calculate ||w - w^t||^2 for each layer
                    # Note: We must move w_global[name] to the current device
                    # before calculating the difference.
                    proximal_term += torch.sum((param - w_global[name].to(device)) ** 2)

            # Add the FedProx term: Loss + (mu/2) * ||w - w^t||^2
            loss += (mu / 2.0) * proximal_term

            # --- C. Backpropagation ---
            loss.backward()
            opt.step()

        # 4. Step the Local Scheduler after each epoch
        # This will adjust the LR for the next epoch's optimization steps.
        local_scheduler.step()
        
        # Optional: Check the decay
        # print(f"  Epoch {epoch+1}/{epochs}: Local LR={local_scheduler.get_last_lr()[0]:.8f}")

    return model.state_dict() # return updated weights

In [10]:
def fed_avg(models):
    avg = copy.deepcopy(models[0])
    for k in avg.keys():
        for i in range(1, len(models)):
            avg[k] += models[i][k]
        avg[k] = avg[k] / len(models)
    return avg

In [11]:
metrics_history = defaultdict(lambda: defaultdict(list))

In [12]:
import copy
import numpy as np
from torch.utils.data import DataLoader, Subset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLIENTS = 3
ROUNDS = 10
LOCAL_EPOCHS = 10
LR = 0.0001
MU = 0.01
NUM_CLASSES = 6
N_FOLDS = 5
SEED = 42
BATCH_SIZE = 32

# --------------------------------------------------------
# Load datasets + create folds for each client
# --------------------------------------------------------
client_datasets = {}
client_folds = {}

for cid in range(1, NUM_CLIENTS + 1):
    ds = get_client_dataset(cid)
    client_datasets[cid] = ds
    folds = get_stratified_folds(ds, n_splits=N_FOLDS, seed=SEED)
    client_folds[cid] = folds
    print(f"Client {cid}: {len(ds)} samples, {len(folds)} folds created.")

# --------------------------------------------------------
# Utility function to compute class weights
# --------------------------------------------------------
def compute_class_weights_from_indices(dataset, indices, num_classes=NUM_CLASSES, device=DEVICE):
    counts = [0] * num_classes
    for i in indices:
        _, lbl = dataset[i]
        counts[lbl] += 1
    max_count = max(c for c in counts if c > 0)
    weights = [(max_count / c) if c > 0 else 0 for c in counts]
    return torch.tensor(weights, dtype=torch.float32).to(device)

# --------------------------------------------------------
# Main results container
# --------------------------------------------------------
fold_metrics_per_client = {cid: [] for cid in range(1, NUM_CLIENTS + 1)}

# --------------------------------------------------------
# 5-FOLD FEDERATED TRAINING
# --------------------------------------------------------
print("\n===== STARTING FULL 5-FOLD FEDERATED VALIDATION =====")

for fold_id in range(N_FOLDS):
    print(f"\n\n===================== FOLD {fold_id+1}/{N_FOLDS} =====================")

    # NEW global model for each fold (your requirement)
    global_model = Net().to(DEVICE)

    # Run a full FL training for this fold
    for r in range(ROUNDS):
        print(f"\n----- Fold {fold_id+1}: Round {r+1}/{ROUNDS} -----")

        client_updates = []

        for cid in range(1, NUM_CLIENTS + 1):

            print(f"\n Client {cid}: Training on fold {fold_id+1}")

            ds = client_datasets[cid]
            fold = client_folds[cid][fold_id]
            train_idx, val_idx = fold

            train_loader = DataLoader(Subset(ds, train_idx), batch_size=BATCH_SIZE, shuffle=True)
            val_loader   = DataLoader(Subset(ds, val_idx),   batch_size=BATCH_SIZE, shuffle=False)

            cw = compute_class_weights_from_indices(ds, train_idx)

            # Local model starts from the fresh global model for this fold
            local_model = copy.deepcopy(global_model)

            # Local FedProx training
            updated_state = local_train_fedprox(
                local_model, train_loader,
                LOCAL_EPOCHS, LR, MU, DEVICE, cw
            )

            client_updates.append(updated_state)

        # FedAvg aggregation
        new_global = fed_avg(client_updates)
        global_model.load_state_dict(new_global)

    # --------------------------
    # AFTER ALL ROUNDS → Evaluate fold
    # --------------------------
    print(f"\n========== Evaluating Fold {fold_id+1} ==========")

    for cid in range(1, NUM_CLIENTS + 1):
        ds = client_datasets[cid]
        _, val_idx = client_folds[cid][fold_id]

        val_loader = DataLoader(Subset(ds, val_idx), batch_size=BATCH_SIZE, shuffle=False)
        cw = compute_class_weights_from_indices(ds, val_idx)

        loss, report = local_eval(global_model, val_loader, DEVICE, cw)

        fold_summary = {
            "loss": loss,
            "accuracy": report["accuracy"],
            "macro_precision": report["macro avg"]["precision"],
            "macro_recall": report["macro avg"]["recall"],
            "macro_f1": report["macro avg"]["f1-score"]
        }

        fold_metrics_per_client[cid].append(fold_summary)

        print(f"\nClient {cid} Fold {fold_id+1} Results:")
        print(f"  Accuracy        : {fold_summary['accuracy']:.4f}")
        print(f"  Macro Precision : {fold_summary['macro_precision']:.4f}")
        print(f"  Macro Recall    : {fold_summary['macro_recall']:.4f}")
        print(f"  Macro F1        : {fold_summary['macro_f1']:.4f}")
        print(f"  Loss            : {fold_summary['loss']:.4f}")

# --------------------------------------------------------
# FINAL MACRO AVERAGE ACROSS ALL 5 FOLDS (PER CLIENT)
# --------------------------------------------------------
print("\n\n=================== FINAL CROSS-FOLD RESULTS ===================")

for cid in range(1, NUM_CLIENTS + 1):
    records = fold_metrics_per_client[cid]

    accs   = [r["accuracy"] for r in records]
    mpres  = [r["macro_precision"] for r in records]
    mrec   = [r["macro_recall"] for r in records]
    mf1    = [r["macro_f1"] for r in records]
    losses = [r["loss"] for r in records]

    print(f"\n------ CLIENT {cid} (Macro Average Across {N_FOLDS} Folds) ------")
    print(f"Accuracy Mean ± Std        : {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"Macro Precision Mean ± Std : {np.mean(mpres):.4f} ± {np.std(mpres):.4f}")
    print(f"Macro Recall Mean ± Std    : {np.mean(mrec):.4f} ± {np.std(mrec):.4f}")
    print(f"Macro F1 Mean ± Std        : {np.mean(mf1):.4f} ± {np.std(mf1):.4f}")
    print(f"Loss Mean ± Std            : {np.mean(losses):.4f} ± {np.std(losses):.4f}")

print("\n=============== DONE ===============\n")

Client 1: 1420 samples, 5 folds created.
Client 2: 1395 samples, 5 folds created.
Client 3: 1387 samples, 5 folds created.

===== STARTING FULL 5-FOLD FEDERATED VALIDATION =====


===================== FOLD 1/5 =====================


Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth
100%|██████████| 47.2M/47.2M [00:00<00:00, 168MB/s] 



----- Fold 1: Round 1/10 -----

 Client 1: Training on fold 1

 Client 2: Training on fold 1

 Client 3: Training on fold 1

----- Fold 1: Round 2/10 -----

 Client 1: Training on fold 1

 Client 2: Training on fold 1

 Client 3: Training on fold 1

----- Fold 1: Round 3/10 -----

 Client 1: Training on fold 1

 Client 2: Training on fold 1

 Client 3: Training on fold 1

----- Fold 1: Round 4/10 -----

 Client 1: Training on fold 1

 Client 2: Training on fold 1

 Client 3: Training on fold 1

----- Fold 1: Round 5/10 -----

 Client 1: Training on fold 1

 Client 2: Training on fold 1

 Client 3: Training on fold 1

----- Fold 1: Round 6/10 -----

 Client 1: Training on fold 1

 Client 2: Training on fold 1

 Client 3: Training on fold 1

----- Fold 1: Round 7/10 -----

 Client 1: Training on fold 1

 Client 2: Training on fold 1

 Client 3: Training on fold 1

----- Fold 1: Round 8/10 -----

 Client 1: Training on fold 1

 Client 2: Training on fold 1

 Client 3: Training on fold 1
